In [ ]:
# === ROGII TPS-surface crash probe (subprocess-isolated; survives segfault/OOM) ===
import subprocess, sys, time

CHILD_SRC = '\nimport sys, time\nimport numpy as np, pandas as pd\nfrom pathlib import Path\n\nprint("[CHILD] start", flush=True)\nDATA = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")\nTRAIN_DIR = DATA / "train"\nFORMATIONS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]\nhw_paths = sorted(TRAIN_DIR.glob("*__horizontal_well.csv"))\ntrain_wids = [p.stem.replace("__horizontal_well", "") for p in hw_paths]\nprint(f"[CHILD] train wells={len(train_wids)} numpy={np.__version__}", flush=True)\n\n_LAM = 5.0\n\n\ndef _tps_U(r2):\n    return np.where(r2 > 1e-12, 0.5 * r2 * np.log(np.maximum(r2, 1e-30)), 0.0)\n\n\nclass _FormSurfTPS:\n    def __init__(self, wids, data_dir):\n        raw = []\n        for wid in wids:\n            p = data_dir / (wid + "__horizontal_well.csv")\n            try:\n                df = pd.read_csv(p, usecols=["X", "Y"] + FORMATIONS).dropna()\n            except Exception:\n                continue\n            if len(df) == 0:\n                continue\n            raw.append((wid, df))\n        W = len(raw)\n        rows_x = []; rows_y = []; rows_w = []; rows_F = {f: [] for f in FORMATIONS}\n        code = {}\n        for wid, df in raw:\n            c = code.setdefault(wid, len(code))\n            rows_x.append(np.array([df["X"].mean()], np.float64))\n            rows_y.append(np.array([df["Y"].mean()], np.float64))\n            rows_w.append(np.full(1, c, np.int32))\n            for f in FORMATIONS:\n                rows_F[f].append(np.array([np.median(df[f].to_numpy(np.float64))], np.float64))\n        self.code = code\n        self.nx = np.concatenate(rows_x); self.ny = np.concatenate(rows_y)\n        self.nw = np.concatenate(rows_w)\n        self.nF = {f: np.concatenate(rows_F[f]) for f in FORMATIONS}\n        N = len(self.nx)\n        self.mx = float(self.nx.mean()); self.my = float(self.ny.mean())\n        self.sx = float(self.nx.std() or 1.0); self.sy = float(self.ny.std() or 1.0)\n        self._Xn = (self.nx - self.mx) / self.sx\n        self._Yn = (self.ny - self.my) / self.sy\n        print(f"[CHILD][SURF-TPS] wells={W} N={N} saddle={(N+3)}x{(N+3)} "\n              f"({(N+3)**2*8/1e6:.0f}MB) lam={_LAM}", flush=True)\n        t0 = time.time(); self._full = self._solve(np.ones(N, bool))\n        print(f"[CHILD][SURF-TPS] full solve {time.time()-t0:.2f}s", flush=True)\n        t0 = time.time(); self._loo = {}\n        uniq = np.unique(self.nw)\n        for ii, c in enumerate(uniq):\n            self._loo[int(c)] = self._solve(self.nw != c)\n            if ii % 100 == 0:\n                print(f"[CHILD][SURF-TPS] LOO {ii}/{len(uniq)} ({time.time()-t0:.0f}s)", flush=True)\n        print(f"[CHILD][SURF-TPS] {len(self._loo)} per-well LOO solves {time.time()-t0:.1f}s", flush=True)\n\n    def _solve(self, mask):\n        Xn = self._Xn[mask]; Yn = self._Yn[mask]; m = len(Xn)\n        dx = Xn[:, None] - Xn[None, :]; dy = Yn[:, None] - Yn[None, :]\n        K = _tps_U(dx * dx + dy * dy) + _LAM * np.eye(m)\n        P = np.column_stack([np.ones(m), Xn, Yn])\n        Asys = np.zeros((m + 3, m + 3))\n        Asys[:m, :m] = K; Asys[:m, m:] = P; Asys[m:, :m] = P.T\n        Asys += 1e-8 * np.eye(m + 3)\n        RHS = np.zeros((m + 3, len(FORMATIONS)))\n        for j, f in enumerate(FORMATIONS):\n            RHS[:m, j] = self.nF[f][mask]\n        SOL = np.linalg.solve(Asys, RHS)\n        coeffs = {f: (SOL[:m, j], SOL[m:, j]) for j, f in enumerate(FORMATIONS)}\n        return {"Xn": Xn, "Yn": Yn, "coeffs": coeffs}\n\n    def _eval(self, bundle, xy, f):\n        Xq = (xy[:, 0] - self.mx) / self.sx\n        Yq = (xy[:, 1] - self.my) / self.sy\n        dx = Xq[:, None] - bundle["Xn"][None, :]\n        dy = Yq[:, None] - bundle["Yn"][None, :]\n        w, a = bundle["coeffs"][f]\n        return _tps_U(dx * dx + dy * dy) @ w + a[0] + a[1] * Xq + a[2] * Yq\n\n    def predict(self, xy, wid=None):\n        xy = np.atleast_2d(xy)\n        code = self.code.get(wid) if wid is not None else None\n        b = self._loo[code] if (code is not None and code in self._loo) else self._full\n        return {f: self._eval(b, xy, f).astype(np.float64) for f in FORMATIONS}\n\n\nt0 = time.time()\n_FS = _FormSurfTPS(train_wids, TRAIN_DIR)\nprint(f"[CHILD] FS built nodes={len(_FS.nx)} ({time.time()-t0:.0f}s)", flush=True)\n\n\ndef _surf_feats(paths, is_train):\n    recs = []\n    for k, p in enumerate(paths):\n        wid = p.stem.replace("__horizontal_well", "")\n        try:\n            hw = pd.read_csv(p)\n        except Exception:\n            continue\n        if "TVT_input" not in hw.columns:\n            continue\n        kn = hw[hw["TVT_input"].notna()]; ev = hw[hw["TVT_input"].isna()]\n        if len(ev) == 0 or len(kn) < 10:\n            continue\n        last_tvt = float(kn["TVT_input"].to_numpy()[-1])\n        swid = wid if is_train else None\n        xk = kn[["X", "Y"]].to_numpy(np.float64); zk = kn["Z"].to_numpy(np.float64)\n        tk = kn["TVT_input"].to_numpy(np.float64)\n        xe = ev[["X", "Y"]].to_numpy(np.float64); ze = ev["Z"].to_numpy(np.float64)\n        Sk = _FS.predict(xk, swid); Se = _FS.predict(xe, swid)\n        tvts = {}; bs = []\n        for f in FORMATIONS:\n            b_f = float(np.median(tk + zk - Sk[f])); bs.append(b_f); tvts[f] = (-ze + Se[f] + b_f)\n        rng = float(tk.max() - tk.min()); cap = 3.0 * rng + 1000.0\n        deltas = {f: np.clip(tvts[f] - last_tvt, -cap, cap) for f in FORMATIONS}\n        ids = [f"{wid}_{i}" for i in ev.index]\n        d = {"id": ids}\n        for f in FORMATIONS:\n            d[f"tvtS_{f}_d"] = deltas[f].astype(np.float32)\n        recs.append(pd.DataFrame(d))\n        if k % 100 == 0:\n            print(f"[CHILD][surf_feats] {k}/{len(paths)}", flush=True)\n    return pd.concat(recs, ignore_index=True) if recs else pd.DataFrame({"id": []})\n\n\nt0 = time.time()\nsf = _surf_feats(hw_paths, True)\nprint(f"[CHILD] surf_feats rows={len(sf)} cols={list(sf.columns)} ({time.time()-t0:.0f}s)", flush=True)\nprint("PROBE_OK", flush=True)\n'

with open("/kaggle/working/tps_probe.py", "w") as _fh:
    _fh.write(CHILD_SRC)

print("=== running TPS-surface build in an ISOLATED subprocess ===", flush=True)
_t0 = time.time()
rc = None; out = ""; err = ""
try:
    _r = subprocess.run([sys.executable, "-u", "/kaggle/working/tps_probe.py"],
                        capture_output=True, text=True, timeout=2400)
    rc, out, err = _r.returncode, _r.stdout, _r.stderr
except subprocess.TimeoutExpired as _e:
    out = _e.stdout if isinstance(_e.stdout, str) else (_e.stdout or b"").decode("utf-8", "replace")
    err = _e.stderr if isinstance(_e.stderr, str) else (_e.stderr or b"").decode("utf-8", "replace")
    print("!!! CHILD TIMED OUT (2400s) !!!", flush=True)

print(f"=== child finished: returncode={rc}  elapsed={time.time()-_t0:.0f}s ===", flush=True)
print("---------------- CHILD STDOUT (tail) ----------------", flush=True)
print(out[-12000:], flush=True)
print("---------------- CHILD STDERR (tail) ----------------", flush=True)
print(err[-12000:], flush=True)
print("-----------------------------------------------------", flush=True)
if rc is None:
    print(">>> VERDICT: child TIMED OUT (surface build too slow or hung)", flush=True)
elif rc < 0:
    print(f">>> VERDICT: child KILLED BY SIGNAL {-rc} "
          f"(segfault/SIGSEGV=11, OOM-kill/SIGKILL=9). The last CHILD STDOUT line "
          f"localizes the death point.", flush=True)
elif rc != 0:
    print(">>> VERDICT: child raised a PYTHON EXCEPTION (see CHILD STDERR traceback).", flush=True)
else:
    print(">>> VERDICT: child COMPLETED OK -- the TPS-surface build is NOT the crash. "
          "Bisect the next stage (merge / downstream pipeline).", flush=True)
print("DONE_PARENT", flush=True)
